### Custom Attention Mechanism Implementation

We will implement a basic **Dot-Product Attention** (often used in Luong attention). The mathematical formulation is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

Where:
*   $Q$ is the Query matrix
*   $K$ is the Key matrix
*   $V$ is the Value matrix
*   $d_k$ is the scaling factor (dimension of keys) to prevent extremely small gradients during softmax when dimensions are large.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DotProductAttention(nn.Module):
    def __init__(self, scale=True):
        super(DotProductAttention, self).__init__()
        self.scale = scale

    def forward(self, query, key, value):
        # query: [batch_size, num_queries, d_k]
        # key:   [batch_size, num_keys, d_k]
        # value: [batch_size, num_keys, d_v]

        # 1. Compute attention scores (dot product of Query and Key)
        # We transpose the last two dimensions of key to perform batch matrix multiplication (bmm)
        scores = torch.bmm(query, key.transpose(1, 2))

        # 2. Optionally scale scores
        if self.scale:
            d_k = query.size(-1)
            scores = scores / (d_k ** 0.5)

        # 3. Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)

        # 4. Multiply attention weights by values
        context = torch.bmm(attention_weights, value)

        return context, attention_weights

In [2]:
# Demonstration with dummy data
if __name__ == "__main__":
    # Dimensions: batch_size=2, sequence_length=3, feature_dim=4
    batch_size = 2
    seq_len = 3
    d_k = 4
    d_v = 4

    # Initialize our custom attention layer
    attention_layer = DotProductAttention(scale=True)

    # Create random Tensors representing Query, Key, and Value
    torch.manual_seed(42)
    Q = torch.randn(batch_size, seq_len, d_k)
    K = torch.randn(batch_size, seq_len, d_k)
    V = torch.randn(batch_size, seq_len, d_v)

    # Run forward pass
    context, weights = attention_layer(Q, K, V)

    print("Query shape:", Q.shape)
    print("Key shape:", K.shape)
    print("Value shape:", V.shape)
    print("\nContext vector (Output) shape:", context.shape)
    print("Attention weights shape:", weights.shape)

    print("\nAttention Weights for Batch 1:\n", weights[0])
    print("Context vector for Batch 1:\n", context[0])

Query shape: torch.Size([2, 3, 4])
Key shape: torch.Size([2, 3, 4])
Value shape: torch.Size([2, 3, 4])

Context vector (Output) shape: torch.Size([2, 3, 4])
Attention weights shape: torch.Size([2, 3, 3])

Attention Weights for Batch 1:
 tensor([[0.2746, 0.2220, 0.5034],
        [0.4423, 0.3890, 0.1687],
        [0.2303, 0.5218, 0.2479]])
Context vector for Batch 1:
 tensor([[-0.2196,  0.3126, -0.0202, -0.3293],
        [ 0.0989,  0.1511, -0.1175, -0.2992],
        [ 0.1140,  0.3381,  0.3332, -0.1965]])
